In [1]:
# Setup and imports
%pip install trackio

import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

import pytorch_lightning as pl
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.metrics import f1_score, mean_squared_error
from sklearn.model_selection import train_test_split
import trackio

# Reproducibility
seed_everything(42, workers=True)

# Device 
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Using device:", DEVICE)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.0/875.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.9 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.4
    Uninstalling pydanti

Seed set to 42


Using device: cuda


In [2]:
# Dataset paths
DATA_DIR = Path("/kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset")
TRAIN_IMAGES_DIR = DATA_DIR / "train"
TEST_IMAGES_DIR = DATA_DIR / "test"
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"

In [3]:
# Define dataset and transforms

# Transforms 
IMG_SIZE = 224  

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# FaceDataset Class
class FaceDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path'] # This comes from the 'full_path' column
        if not Path(img_path).is_absolute():
            # Combines root_dir with path from CSV
            # e.g., './face_dataset' + 'train/00000.jpg'
            img_path = self.root_dir / img_path

        try:
            img = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"Error: Image file not found at {img_path}")
            # Skip this item and try the next one
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            img = self.transform(img)

        sample = {'image': img, 'id': row['id']}
        if not self.is_test:
            sample['age'] = torch.tensor(row['age'], dtype=torch.float32)
            sample['gender'] = torch.tensor(row['gender'], dtype=torch.float32)
        return sample

# FaceDataModule Class
class FaceDataModule(pl.LightningDataModule):
    def __init__(self, train_df, val_df, test_df, root_dir, batch_size=32, num_workers=4):
        super().__init__()
        self.train_df, self.val_df, self.test_df = train_df, val_df, test_df
        self.root_dir = root_dir
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        self.train_ds = FaceDataset(self.train_df, self.root_dir, transform=train_transforms)
        self.val_ds = FaceDataset(self.val_df, self.root_dir, transform=val_transforms)
        self.test_ds = FaceDataset(self.test_df, self.root_dir, transform=val_transforms, is_test=True)
        self.age_min = float(self.train_df['age'].min())
        self.age_max = float(self.train_df['age'].max())

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=3, n_filters=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, n_filters, 3, padding=1), nn.BatchNorm2d(n_filters), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(n_filters, n_filters*2, 3, padding=1), nn.BatchNorm2d(n_filters*2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(n_filters*2, n_filters*4, 3, padding=1), nn.BatchNorm2d(n_filters*4), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(n_filters*4, n_filters*8, 3, padding=1), nn.BatchNorm2d(n_filters*8), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1))
        )
        feat_dim = n_filters*8
        self.age_head = nn.Sequential(nn.Flatten(), nn.Linear(feat_dim, 128), nn.ReLU(), nn.Linear(128,1))
        self.gender_head = nn.Sequential(nn.Flatten(), nn.Linear(feat_dim,128), nn.ReLU(), nn.Linear(128,1))

    def forward(self,x):
        f = self.conv(x)
        return {'age': self.age_head(f).squeeze(1), 'gender_logit': self.gender_head(f).squeeze(1)}

In [5]:
'''
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger
import pytorch_lightning as pl
from torchvision import models 
from huggingface_hub import HfApi 
from kaggle_secrets import UserSecretsClient

# Defining data paths
DATA_ROOT = Path("./face_dataset")
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_WRITE_TOKEN")

# Check if directories exist
if not (DATA_ROOT.exists() and (DATA_ROOT / "train").exists() and (DATA_ROOT / "test").exists()):
    print(f"Error: Directory not found or is missing 'train'/'test' subfolders: {DATA_ROOT}")
    print("Please make sure your 'face_dataset' folder is in the correct location.")
else:
    print(f"Using data from: {DATA_ROOT}")

    # Load Datasets from CSVs 
    try:
        train_full_df = pd.read_csv(DATA_ROOT / "train.csv")
        test_df = pd.read_csv(DATA_ROOT / "test.csv")
        print("Loaded train.csv and test.csv")
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print(f"Make sure 'train.csv' and 'test.csv' are inside {DATA_ROOT}")
        raise

    # Rename 'full_path' to 'image_path' 
    train_full_df = train_full_df.rename(columns={'full_path': 'image_path'})
    test_df = test_df.rename(columns={'full_path': 'image_path'})
    print("Renamed 'full_path' column to 'image_path'")

    # Create Validation Split 
    train_df, val_df = train_test_split(train_full_df, test_size=0.1, random_state=42)
    print(f"Data split: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test")

    # Instantiate DataModule 
    dm = FaceDataModule(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        root_dir=DATA_ROOT, 
        batch_size=32,
        num_workers=0 
    )
    
    dm.setup()
    print(f"DataModule setup. Using age range: {dm.age_min} - {dm.age_max}")

    # Instantiate Model
    print("Loading Scratch Model: SimpleCNN()")
    model = SimpleCNN() 

    # Wrap in LightningModule
    lit_model = MultiTaskLitModule(
        model,
        lr=1e-4, # Using the same LR as your fine-tuning
        weight_decay=1e-5,
        age_min=dm.age_min,
        age_max=dm.age_max
    )

    # Configure Trainer
    checkpoint_callback = ModelCheckpoint(
        monitor='val_rmse', 
        mode='min',         
        dirpath='./',       
        filename='cnn_model_checkpoint' 
    )
    
    early_stop_callback = EarlyStopping(
        monitor='val_rmse',
        patience=3,
        verbose=True,
        mode='min'
    )
    
    logger = CSVLogger("logs", name="face_model_scratch_cnn") 

    trainer = pl.Trainer(
        max_epochs=15, 
        accelerator=DEVICE, 
        devices=1,
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback]
    )

    # Run Training 
    print("--- Starting Model Training (Scratch CNN) ---")

    # Define the checkpoint path you want to resume from
    resume_ckpt_path = "checkpoints/cnn_model_checkpoint.ckpt"

    if os.path.exists(resume_ckpt_path):
        print(f"Resuming training from checkpoint: {resume_ckpt_path}")
        # Pass the ckpt_path to trainer.fit()
        trainer.fit(lit_model, datamodule=dm, ckpt_path=resume_ckpt_path)
    else:
        print(f"Starting a new training run (Checkpoint not found at: {resume_ckpt_path})")
        trainer.fit(lit_model, datamodule=dm)
        
    print("--- Training Complete ---")

    # Save Final Model for Inference Cell 
    print(f"Loading best model from: {checkpoint_callback.best_model_path}")
    
    best_lit_model = MultiTaskLitModule.load_from_checkpoint(
        checkpoint_callback.best_model_path,
        model=SimpleCNN(), 
        age_min=dm.age_min,
        age_max=dm.age_max
    )

    torch.save(best_lit_model.model.state_dict(), 'cnn_scratch_final.pth')
    print("Saved best model weights to 'cnn_scratch_final.pth'")

    print("\n--- Starting Hugging Face Model Upload ---")

    # HF_TOKEN = os.environ.get('HF_WRITE_TOKEN')


    if not HF_TOKEN:
        print("HF_WRITE_TOKEN not found in environment variables.")
        print("Skipping automatic model upload.")
        print("Please upload 'resnet_finetuned_final.pth' to your Space manually.")
    else:
        try:
            REPO_ID = "jananiramaseshan/dlgenai-nppe"
            LOCAL_FILE_PATH = "cnn_scratch_final.pth"
            REPO_FILE_PATH = "cnn_scratch_final.pth"

            print(f"Uploading {LOCAL_FILE_PATH} to {REPO_ID}...")
            
            api = HfApi(token=HF_TOKEN) 
            api.upload_file(
                path_or_fileobj=LOCAL_FILE_PATH,
                path_in_repo=REPO_FILE_PATH,
                repo_id=REPO_ID,
                repo_type="space"
            )
            
            print("--- Upload Complete! ---")
            print("Your Hugging Face Space will now automatically restart with the new model.")

        except Exception as e:
            print("--- Hugging Face Upload Failed ---")
            print(f"Error: {e}")
            print("Please upload 'resnet_finetuned_final.pth' to your Space manually.")

    print("\nCell 7 is finished.")
    '''

'\nimport pandas as pd\nfrom pathlib import Path\nfrom sklearn.model_selection import train_test_split\nfrom pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping\nfrom pytorch_lightning.loggers import CSVLogger\nimport pytorch_lightning as pl\nfrom torchvision import models \nfrom huggingface_hub import HfApi \nfrom kaggle_secrets import UserSecretsClient\n\n# Defining data paths\nDATA_ROOT = Path("./face_dataset")\nuser_secrets = UserSecretsClient()\nHF_TOKEN = user_secrets.get_secret("HF_WRITE_TOKEN")\n\n# Check if directories exist\nif not (DATA_ROOT.exists() and (DATA_ROOT / "train").exists() and (DATA_ROOT / "test").exists()):\n    print(f"Error: Directory not found or is missing \'train\'/\'test\' subfolders: {DATA_ROOT}")\n    print("Please make sure your \'face_dataset\' folder is in the correct location.")\nelse:\n    print(f"Using data from: {DATA_ROOT}")\n\n    # Load Datasets from CSVs \n    try:\n        train_full_df = pd.read_csv(DATA_ROOT / "train.csv")\n  

In [6]:
def get_finetuned_resnet(weights=None):
    base_model = models.resnet18(weights=weights)
    feat_dim = base_model.fc.in_features
    base_model.fc = nn.Identity()

    class ResnetMultiHead(nn.Module):
        def __init__(self, base_model, feat_dim):
            super().__init__()
            self.backbone = base_model
            self.age_head = nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.25), nn.Linear(256,1))
            self.gender_head = nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.25), nn.Linear(256,1))
        def forward(self,x):
            f = self.backbone(x)
            return {'age': self.age_head(f).squeeze(1), 'gender_logit': self.gender_head(f).squeeze(1)}
    return ResnetMultiHead(base_model, feat_dim)

In [7]:
class MultiTaskLitModule(pl.LightningModule):
    # --- MODIFIED to accept trackio_run ---
    def __init__(self, model, lr=3e-3, weight_decay=1e-4, age_min=0, age_max=100, trackio_run=None):
        super().__init__()
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay
        self.age_criterion = nn.MSELoss()
        self.gender_criterion = nn.BCEWithLogitsLoss()
        self.age_min, self.age_max = age_min, age_max
        self.trackio_run = trackio_run  # Store the run object
        
        # Used to aggregate validation step outputs
        self.validation_step_outputs = []

    def forward(self,x): return self.model(x)

    def training_step(self,batch,batch_idx):
        out = self(batch['image'])
        
        # Calculate losses separately
        loss_age = self.age_criterion(out['age'], batch['age'])
        loss_gender = self.gender_criterion(out['gender_logit'], batch['gender'])
        
        # --- Hyperparameter to Tune ---
        # Give gender loss 10x more importance
        gender_weight = 10.0  # Try 5.0, 10.0, or 20.0
        
        loss = loss_age + (loss_gender * gender_weight)
        
        # Log all three so you can see them in TrackIO
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_loss_age', loss_age, on_step=False, on_epoch=True)
        self.log('train_loss_gender', loss_gender, on_step=False, on_epoch=True)
        
        return loss

    def validation_step(self,batch,batch_idx):
        out = self(batch['image'])
        age_pred = out['age'].detach().cpu()
        age_true = batch['age'].detach().cpu()
        gender_pred_prob = torch.sigmoid(out['gender_logit']).detach().cpu()
        gender_true = batch['gender'].detach().cpu().int()

        # Store outputs for epoch-end calculation
        self.validation_step_outputs.append({
            'age_true': age_true, 'age_pred': age_pred,
            'gender_true': gender_true, 'gender_pred_prob': gender_pred_prob
        })
        return {} # No need to return anything here

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        
        # Use a "smart" scheduler that monitors your validation score
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='min',      # We want to minimize 'val_rmse'
            factor=0.2,      # Reduce LR by 80% (0.2x)
            patience=2,      # Wait 2 epochs for improvement before cutting
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_rmse", # Tell Lightning what to monitor
            },
        }

    # --- MODIFIED to calculate metrics and log to TrackIO ---
    def on_validation_epoch_end(self):
        # Concatenate all batch outputs
        all_age_true = torch.cat([x['age_true'] for x in self.validation_step_outputs]).numpy()
        all_age_pred = torch.cat([x['age_pred'] for x in self.validation_step_outputs]).numpy()
        all_gender_true = torch.cat([x['gender_true'] for x in self.validation_step_outputs]).numpy()
        all_gender_prob = torch.cat([x['gender_pred_prob'] for x in self.validation_step_outputs]).numpy()
        all_gender_pred = (all_gender_prob >= 0.5).astype(int)

        # Clear the list for the next epoch
        self.validation_step_outputs.clear()

        # Calculate metrics
        rmse = np.sqrt(mean_squared_error(all_age_true, all_age_pred))
        macro_f1 = f1_score(all_gender_true, all_gender_pred, average='macro', zero_division=0)
        nrmse = rmse / max(1e-8, (self.age_max - self.age_min))

        # Log metrics to PyTorch Lightning
        self.log_dict({'val_rmse': rmse, 'val_nrmse': nrmse, 'val_f1': macro_f1}, prog_bar=True)

        # --- MODIFIED to send metrics to TrackIO ---
        if self.trackio_run:
            epoch = self.current_epoch
            metrics_to_log = {
                'val_rmse': rmse,
                'val_nrmse': nrmse,
                'val_f1': macro_f1
            }
            # Also log train_loss if available
            if 'train_loss' in self.trainer.logged_metrics:
                metrics_to_log['train_loss'] = self.trainer.logged_metrics['train_loss'].item()

            try:
                # Use trackio_run.log() for logging metrics
                self.trackio_run.log(metrics_to_log, step=epoch)
            except Exception as e:
                print(f"Failed to log to TrackIO: {e}")

In [8]:
import trackio
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_WRITE_TOKEN")
# HF_WRITE_TOKEN_VAL = os.environ.get('HF_WRITE_TOKEN', None)

trackio_run = None

if HF_TOKEN:
    try:
        # Authenticate with Hugging Face first
        from huggingface_hub import login
        login(token=HF_TOKEN)
        
        # Initialize TrackIO with space_id parameter
        trackio_run = trackio.init(
            project='dlgenai-nppe',
            name='resnet18_finetune_run_1',
            config={
                'model': 'resnet18',
                'batch_size': 32,
                'lr': 3e-3,
                'max_epochs': 15,
                'device': DEVICE
            },
            space_id='jananiramaseshan/dlgenai-nppe'
        )
        
        # Your dashboard is available at your Space URL
        print("TrackIO run created successfully!")
        print("View your dashboard at: https://huggingface.co/spaces/jananiramaseshan/dlgenai-nppe")
        
    except Exception as e:
        print(f"TrackIO init failed: {e}")
        print("Continuing without TrackIO logging.")
else:
    print("HF_WRITE_TOKEN not found. Skipping TrackIO init.")


* Trackio project initialized: dlgenai-nppe
* Trackio metrics will be synced to Hugging Face Dataset: jananiramaseshan/dlgenai-nppe-dataset
* Found existing space: https://huggingface.co/spaces/jananiramaseshan/dlgenai-nppe
* View dashboard by going to: https://jananiramaseshan-dlgenai-nppe.hf.space/


* Created new run: resnet18_finetune_run_1
TrackIO run created successfully!
View your dashboard at: https://huggingface.co/spaces/jananiramaseshan/dlgenai-nppe


In [9]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger
import pytorch_lightning as pl
from torchvision import models # Make sure models is imported
from huggingface_hub import HfApi # Import for HF upload
from kaggle_secrets import UserSecretsClient

# Dataset paths (Kaggle)
DATA_ROOT = Path("/kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset")
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_WRITE_TOKEN")


# Check if directories exist
if not (DATA_ROOT.exists() and (DATA_ROOT / "train").exists() and (DATA_ROOT / "test").exists()):
    print(f"Error: Directory not found or is missing 'train'/'test' subfolders: {DATA_ROOT}")
    print("Please make sure your 'face_dataset' folder is in the correct location.")
else:
    print(f"Using data from: {DATA_ROOT}")

    # 2. --- Load Datasets from CSVs ---
    try:
        # Load CSVs from *inside* the data_root folder
        train_full_df = pd.read_csv(DATA_ROOT / "train.csv")
        test_df = pd.read_csv(DATA_ROOT / "test.csv")
        print("Loaded train.csv and test.csv")
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print(f"Make sure 'train.csv' and 'test.csv' are inside {DATA_ROOT}")
        raise

    # 3. --- Rename 'full_path' to 'image_path' ---
    train_full_df = train_full_df.rename(columns={'full_path': 'image_path'})
    test_df = test_df.rename(columns={'full_path': 'image_path'})
    print("Renamed 'full_path' column to 'image_path'")

    # 4. --- Create Validation Split ---
    train_df, val_df = train_test_split(train_full_df, test_size=0.1, random_state=42)
    print(f"Data split: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test")

    # 5. --- Instantiate DataModule ---
    dm = FaceDataModule(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        root_dir=DATA_ROOT, # FaceDataset will combine this with 'image_path'
        batch_size=32,
        num_workers=4 # --- SET TO 0 TO AVOID MAC CRASH ---
    )
    
    dm.setup()
    print(f"DataModule setup. Using age range: {dm.age_min} - {dm.age_max}")

    # 6. --- Instantiate Model ---
    print("Loading pre-trained ResNet18 model...")
    model = get_finetuned_resnet(weights=models.ResNet18_Weights.DEFAULT)

    # Wrap in LightningModule
    lit_model = MultiTaskLitModule(
        model,
        lr=1e-4, # Smaller learning rate for fine-tuning
        weight_decay=1e-5,
        age_min=dm.age_min,
        age_max=dm.age_max
    )

    # 7. --- Configure Trainer ---
    checkpoint_callback = ModelCheckpoint(
        monitor='val_rmse', # Metric from your validation_step
        mode='min',         # Minimize RMSE
        dirpath='./',       # Save in the root
        filename='best_model_checkpoint' # Temporary name
    )
    
    early_stop_callback = EarlyStopping(
        monitor='val_rmse',
        patience=3,
        verbose=True,
        mode='min'
    )
    
    logger = CSVLogger("logs", name="face_model_finetune")

    trainer = pl.Trainer(
        max_epochs=15, 
        accelerator=DEVICE, # --- Will use "mps" on your Mac ---
        devices=1,
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback]
    )

    # --- 8. Run Training (MODIFIED FOR RESUME) ---
    print("--- Starting Model Training ---")

    # Define the checkpoint path you want to resume from
    resume_ckpt_path = "/kaggle/working/best_model_checkpoint.ckpt"

    if os.path.exists(resume_ckpt_path):
        print(f"Resuming training from checkpoint: {resume_ckpt_path}")
        # Pass the ckpt_path to trainer.fit()
        trainer.fit(lit_model, datamodule=dm, ckpt_path=resume_ckpt_path)
    else:
        print(f"Starting a new training run (Checkpoint not found at: {resume_ckpt_path})")
        trainer.fit(lit_model, datamodule=dm)
        
    print("--- Training Complete ---")

    # 9. --- Save Final Model for Inference Cell ---
    print(f"Loading best model from: {checkpoint_callback.best_model_path}")
    best_lit_model = MultiTaskLitModule.load_from_checkpoint(
        checkpoint_callback.best_model_path,
        model=get_finetuned_resnet(weights=None), 
        age_min=dm.age_min,
        age_max=dm.age_max
    )

    torch.save(best_lit_model.model.state_dict(), 'resnet_finetuned_final.pth')
    print("Saved best model weights to 'resnet_finetuned_final.pth'")

    # --- 10. Automatic upload to Hugging Face (Kept) ---
    print("\n--- Starting Hugging Face Model Upload ---")

    # HF_TOKEN = os.environ.get('HF_WRITE_TOKEN')

    if not HF_TOKEN:
        print("HF_WRITE_TOKEN not found in environment variables.")
        print("Skipping automatic model upload.")
        print("Please upload 'resnet_finetuned_final.pth' to your Space manually.")
    else:
        try:
            REPO_ID = "jananiramaseshan/dlgenai-nppe"
            LOCAL_FILE_PATH = "resnet_finetuned_final.pth"
            REPO_FILE_PATH = "resnet_finetuned_final.pth"

            print(f"Uploading {LOCAL_FILE_PATH} to {REPO_ID}...")
            
            api = HfApi(token=HF_TOKEN) 
            api.upload_file(
                path_or_fileobj=LOCAL_FILE_PATH,
                path_in_repo=REPO_FILE_PATH,
                repo_id=REPO_ID,
                repo_type="space"
            )
            
            print("--- Upload Complete! ---")
            print("Your Hugging Face Space will now automatically restart with the new model.")

        except Exception as e:
            print("--- Hugging Face Upload Failed ---")
            print(f"Error: {e}")
            print("Please upload 'resnet_finetuned_final.pth' to your Space manually.")

    print("\nCell 7 is finished.")

Using data from: /kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset
Loaded train.csv and test.csv
Renamed 'full_path' column to 'image_path'
Data split: 31237 train, 3471 val, 8677 test
DataModule setup. Using age range: 1.0 - 100.0
Loading pre-trained ResNet18 model...


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 149MB/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


--- Starting Model Training ---
Starting a new training run (Checkpoint not found at: /kaggle/working/best_model_checkpoint.ckpt)


/usr/local/lib/python3.11/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /kaggle/working exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name             | Type              | Params | Mode 
---------------------------------------------------------------
0 | model            | ResnetMultiHead   | 11.4 M | train
1 | age_criterion    | MSELoss           | 0      | train
2 | gender_criterion | BCEWithLogitsLoss | 0      | train
---------------------------------------------------------------
11.4 M    Trainable params
0         Non-trainable params
11.4 M    Total params
45.759    Total estimated model params size (MB)
81        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Monitored metric val_rmse did not improve in the last 3 records. Best score: 8.154. Signaling Trainer to stop.


--- Training Complete ---
Loading best model from: /kaggle/working/best_model_checkpoint.ckpt
Saved best model weights to 'resnet_finetuned_final.pth'

--- Starting Hugging Face Model Upload ---
Uploading resnet_finetuned_final.pth to jananiramaseshan/dlgenai-nppe...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

--- Upload Complete! ---
Your Hugging Face Space will now automatically restart with the new model.

Cell 7 is finished.


In [10]:

model = get_finetuned_resnet(weights=None)
model.load_state_dict(torch.load('resnet_finetuned_final.pth', map_location=DEVICE))
model.to(DEVICE).eval()

test_loader = dm.test_dataloader()
all_preds, all_ids = [], []

print("Starting inference...")
with torch.no_grad():
    for batch in test_loader:
        imgs = batch['image'].to(DEVICE)
        out = model(imgs)
        ages = out['age'].cpu().numpy()
        genders = (torch.sigmoid(out['gender_logit']).cpu().numpy()>=0.5).astype(int)
        all_preds.append(np.stack([ages,genders],axis=1))
        all_ids.extend(batch['id'].tolist())

submission = pd.DataFrame(np.vstack(all_preds), columns=['age','gender'])
submission['id'] = all_ids
# The 'dm' object has the correct age_min and age_max
submission['age'] = submission['age'].clip(dm.age_min, dm.age_max).round(1)
submission['gender'] = submission['gender'].astype(int)
submission = submission[['id','age','gender']].sort_values('id').reset_index(drop=True)

submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")
print("\nSubmission preview:")
print(submission.head())

Starting inference...
Saved submission.csv

Submission preview:
   id   age  gender
0   0  37.5       1
1   1  22.6       0
2   2  38.7       1
3   3  33.9       1
4   4  65.6       1
